# 2. SQL Analysis

## 2.1. Data Loading

Install and import all required R libraries

In [1]:
install.packages(c("sqldf", "googledrive"), quiet = TRUE)

library(sqldf)
library(googledrive)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite



Mount Google Drive to read the cleaned dataset

In [2]:
drive_auth(use_oob=TRUE)

Is it OK to cache OAuth access credentials in the folder ~/.cache/gargle
between R sessions?
1: Yes
2: No
Enter a number between 1 and 2, or enter 0 to exit.
Please point your browser to the following url: 

https://accounts.google.com/o/oauth2/v2/auth?client_id=603366585132-frjlouoa3s2ono25d2l9ukvhlsrlnr7k.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email&redirect_uri=https%3A%2F%2Fwww.tidyverse.org%2Fgoogle-callback%2F&response_type=code&state=c64c23126141b3f11d250fc63ab3a9f2&access_type=offline&prompt=consent



In [4]:
base <- "North Star Case Study Dataset/cleaned/"

dl <- function(name) {
  drive_download(paste0(base, name), overwrite = TRUE)$local_path
}

# NOTE: stringsAsFactors tells R to keep text columns as strings, rather than converting them into categories
hubs <- read.csv(dl("hubs_clean.csv"), stringsAsFactors=FALSE)
customers <- read.csv(dl("customers_clean.csv"), stringsAsFactors=FALSE)
drivers <- read.csv(dl("drivers_clean.csv"), stringsAsFactors=FALSE)
vehicles <- read.csv(dl("vehicles_clean.csv"), stringsAsFactors=FALSE)
orders <- read.csv(dl("orders_clean.csv"), stringsAsFactors=FALSE)
deliveries <- read.csv(dl("deliveries_clean.csv"), stringsAsFactors=FALSE)
incidents <- read.csv(dl("incidents_clean.csv"), stringsAsFactors=FALSE)
complaints <- read.csv(dl("complaints_clean.csv"), stringsAsFactors=FALSE)
app_events <- read.csv(dl("app_events_clean.csv"), stringsAsFactors=FALSE)

File downloaded:

• hubs_clean.csv <id: 1DYLnMQnsQDEGyg1nl93Nxk0FoFBX-5Rk>

Saved locally as:

• hubs_clean.csv

File downloaded:

• customers_clean.csv <id: 1wvGKg18PTzOXvS3zgF7edFBumiPrKeo3>

Saved locally as:

• customers_clean.csv

File downloaded:

• drivers_clean.csv <id: 1ZGAHu6PfaoA1KVod2Cb318kH86o617z2>

Saved locally as:

• drivers_clean.csv

File downloaded:

• vehicles_clean.csv <id: 1JMEYq8khDKpkNLK9BoYFy42DSa8sL1qy>

Saved locally as:

• vehicles_clean.csv

File downloaded:

• orders_clean.csv <id: 13WWlNXhsfPXwHMTqwLxA16lrdcufljuB>

Saved locally as:

• orders_clean.csv

File downloaded:

• deliveries_clean.csv <id: 1i4v-y1YnzmtW9NJ-keXHiStL8eMKdqIV>

Saved locally as:

• deliveries_clean.csv

File downloaded:

• incidents_clean.csv <id: 1sqHskG5ZMhfTodFiSwuj53zU-Az892Ds>

Saved locally as:

• incidents_clean.csv

File downloaded:

• complaints_clean.csv <id: 11RcmLsZhFaydACSzhLnzPA2ZUHmL8ogD>

Saved locally as:

• complaints_clean.csv

File downloaded:

• app_events_cle

## 2.2. Delivery Status Breakdown

Count the number of each delivery status

In [5]:
sqldf("
  SELECT delivery_status,
         COUNT(*) AS total
  FROM deliveries
  GROUP BY delivery_status
  ORDER BY total DESC
")

delivery_status,total
<chr>,<int>
OnTime,616
Delayed,202
Failed,132


## 2.3. Complaint Volume by Type

Which complaint categories are raised most often

In [6]:
sqldf("
  SELECT complaint_type,
         COUNT(*) AS total
  FROM complaints
  GROUP BY complaint_type
  ORDER BY total DESC
")

complaint_type,total
<chr>,<int>
Delay,101
MissedPickup,64
AppIssue,53
DriverBehaviour,51
SupportExperience,20
Billing,16
Damage,15


## 2.4. Hub Performance Summary

Per hub, determine delivery volume, failure count, failure rate and average customer rating

In [7]:
sqldf("
  SELECT h.hub_name,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100 / COUNT(*), 1) AS failed_pct,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_name
  ORDER BY failed_pct DESC
")

hub_name,total_deliveries,failed,failed_pct,avg_rating
<chr>,<int>,<int>,<dbl>,<dbl>
Midtown Relay,128,26,20,3.88
Central Core,115,23,20,3.67
Airport Hub,104,15,14,3.88
West Gate,127,16,12,3.92
Riverside Hub,115,14,12,3.88
North Exchange,136,17,12,3.84
South Link,106,10,9,3.95
East Dock,119,11,9,3.90


## 2.5. Failed Delivery Rate by Zone

Which pickup zones have the worst faliure rate

In [8]:
sqldf("
  SELECT o.pickup_zone,
         COUNT(*) AS total,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100 / COUNT(*), 1) AS failed_pct
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY failed_pct DESC
")

pickup_zone,total,failed,failed_pct
<chr>,<int>,<int>,<dbl>
Central,174,33,18
North,135,22,16
Riverside,119,18,15
West,114,14,12
East,156,19,12
South,139,14,10
Airport,113,12,10


## 2.6. Service Type Complaint Rate

Determine complaint rate for each service type, to find out which service type generates the most dissatisfaction

In [9]:
sqldf("
  SELECT o.service_type,
         COUNT(DISTINCT o.order_id) AS orders,
         COUNT(DISTINCT cp.complaint_id) AS complaints,
         ROUND(COUNT(DISTINCT cp.complaint_id) * 100.0 / COUNT(DISTINCT o.order_id), 1) AS complaint_rate_pct
  FROM orders o
  LEFT JOIN complaints cp ON o.order_id = cp.order_id
  GROUP BY o.service_type
  ORDER BY complaint_rate_pct DESC
")

service_type,orders,complaints,complaint_rate_pct
<chr>,<int>,<int>,<dbl>
Retail,297,83,27.9
Medical,139,37,26.6
Parcel,308,77,25.0
Passenger,341,84,24.6
Business,165,39,23.6


## 2.7. Drive Training vs. Route Overrides

Determine whether it is undertrained drivers that override routes more often

In [10]:
sqldf("
  SELECT CASE
           WHEN dr.training_score < 60  THEN 'Low (<60)'
           WHEN dr.training_score <= 75 THEN 'Mid (60-75)'
           ELSE 'High (>75)'
         END AS training_band,
         COUNT(DISTINCT dr.driver_id) AS drivers,
         ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY training_band
  ORDER BY avg_overrides DESC
")

training_band,drivers,avg_overrides
<chr>,<int>,<dbl>
Mid (60-75),65,1.00
High (>75),89,0.96
Low (<60),16,0.90


## 2.8. Incidents by Vehicle Type

Determine incidents per 100 deliveries in each vehicle type

In [11]:
sqldf("
  SELECT v.vehicle_type,
         COUNT(DISTINCT i.incident_id) AS incidents,
         COUNT(DISTINCT d.delivery_id) AS deliveries,
         ROUND(COUNT(DISTINCT i.incident_id) * 100.0 / COUNT(DISTINCT d.delivery_id), 1) AS incidents_per_100
  FROM deliveries d
  JOIN vehicles v  ON d.vehicle_id = v.vehicle_id
  LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
  GROUP BY v.vehicle_type
  ORDER BY incidents_per_100 DESC
")

vehicle_type,incidents,deliveries,incidents_per_100
<chr>,<int>,<int>,<dbl>
Diesel,47,144,32.6
CargoVan,67,223,30.0
EV,100,339,29.5
Hybrid,66,244,27.0


## 2.9. Monthly Operational Trend

Determine order volume, failed deliveries and complaints per month

In [12]:
sqldf("
  SELECT strftime('%Y-%m', o.order_created_at) AS month,
         COUNT(DISTINCT o.order_id) AS orders,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
         COUNT(DISTINCT cp.complaint_id) AS complaints
  FROM orders o
  LEFT JOIN deliveries d  ON o.order_id = d.order_id
  LEFT JOIN complaints cp ON o.order_id = cp.order_id
  GROUP BY month
  ORDER BY month
")

month,orders,failed_deliveries,complaints
<chr>,<int>,<int>,<int>
2024-01,56,1,17
2024-02,59,7,16
2024-03,65,5,17
2024-04,46,6,15
2024-05,45,2,8
2024-06,60,8,18
2024-07,45,6,12
2024-08,56,4,16
2024-09,60,6,16


## 2.10. Deliveries with both Incidents and Complaints

Determine deliveries that are logged as resolved in one system, and an exception in another

In [13]:
sqldf("
  SELECT d.delivery_id,
         d.order_id,
         d.delivery_status,
         i.incident_type,
         i.severity AS incident_severity,
         cp.complaint_type,
         cp.severity AS complaint_severity
  FROM deliveries d
  JOIN incidents i ON d.delivery_id = i.delivery_id
  JOIN complaints cp ON d.order_id = cp.order_id
  ORDER BY d.delivery_id
")

delivery_id,order_id,delivery_status,incident_type,incident_severity,complaint_type,complaint_severity
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DL00011,O00202,OnTime,ProofMissing,Medium,DriverBehaviour,Medium
DL00036,O00652,OnTime,AppSyncError,Low,Delay,Medium
DL00073,O00421,OnTime,TemperatureIssue,Critical,DriverBehaviour,Medium
DL00148,O00817,OnTime,ProofMissing,Medium,Delay,Medium
DL00189,O01227,Failed,CustomerNoShow,High,DriverBehaviour,Medium
DL00221,O00594,OnTime,BatteryAlert,Medium,DriverBehaviour,High
DL00225,O00056,OnTime,ProofMissing,Medium,SupportExperience,Medium
DL00288,O00636,Delayed,CustomerNoShow,Medium,Billing,Medium
DL00291,O00485,OnTime,ProofMissing,Low,Delay,Medium


## 2.11. Zone-Level Operational Risk

Output total, failed, delayed, missing and complained orders by pickup zone for high-level overview

In [14]:
sqldf("
  SELECT o.pickup_zone,
         COUNT(DISTINCT o.order_id) AS total_orders,
         SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END) AS failed,
         SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
         SUM(COALESCE(d.proof_of_completion_missing, 0)) AS proof_missing,
         COUNT(DISTINCT cp.complaint_id) AS complaints,
         ROUND(AVG(COALESCE(cp.compensation_amount, 0)), 2) AS avg_compensation
  FROM orders o
  LEFT JOIN deliveries d ON o.order_id = d.order_id
  LEFT JOIN complaints cp ON o.order_id = cp.order_id
  GROUP BY o.pickup_zone
  ORDER BY (failed + delayed + proof_missing) DESC
")

pickup_zone,total_orders,failed,delayed,proof_missing,complaints,avg_compensation
<chr>,<int>,<int>,<int>,<int>,<int>,<dbl>
Central,238,33,51,17,60,5.19
East,207,20,31,11,50,4.80
North,174,22,22,12,53,6.34
Riverside,151,19,26,8,45,5.26
Airport,144,12,31,5,32,3.01
West,155,14,22,11,34,4.48
South,181,14,22,7,46,4.11
